# Grados Día Acumulados (GDD)

Los **Growing Degree Days** (GDD) son una métrica agronómica que estima la acumulación de calor
a lo largo de la temporada. Se calcula como:

$$\text{GDD}_{\text{día}} = \max\left(\frac{T_{\max} + T_{\min}}{2} - T_{\text{base}},\; 0\right)$$

Donde $T_{\text{base}}$ depende del cultivo (típicamente 10°C para vid).

In [ ]:
import nekazari as nkz
import pandas as pd

# Descargar temperatura de una estación meteorológica o sensor
# Ajusta entity_urn o device_id a tu entidad real
df = await nkz.timeseries.get_dataframe(
    device_id="120786a0cf364796",
    attributes=["airTemperature"],
    start_date="2026-01-01",
    end_date="2026-03-28",
    resolution=2000,
)

print(f"Registros: {len(df)}")
df.head()

In [ ]:
# Convertir timestamp (epoch seconds) a datetime y agregar por día
df['datetime'] = pd.to_datetime(df['value_0'], unit='s', utc=True) if 'timestamp' not in df.columns else pd.to_datetime(df['timestamp'], unit='s', utc=True)
temp_col = [c for c in df.columns if c.startswith('value_')][0]
df['date'] = df['datetime'].dt.date

daily = df.groupby('date').agg(
    temp_max=(temp_col, 'max'),
    temp_min=(temp_col, 'min'),
).reset_index()

daily.head()

In [ ]:
# Calcular GDD diarios y acumulados
T_BASE = 10.0  # °C — ajustar según cultivo

daily['gdd_day'] = ((daily['temp_max'] + daily['temp_min']) / 2 - T_BASE).clip(lower=0)
daily['gdd_cumulative'] = daily['gdd_day'].cumsum()

print(f"GDD acumulados al {daily['date'].iloc[-1]}: {daily['gdd_cumulative'].iloc[-1]:.1f}")
daily.tail(10)

## Interpretación

| Cultivo | GDD para floración | GDD para cosecha |
|---------|-------------------|------------------|
| Vid (Tempranillo) | ~300-400 | ~1600-1800 |
| Olivo | ~200-300 | ~1200-1500 |
| Trigo | ~150-200 | ~1500-1700 |

Compara tu GDD acumulado con estos umbrales para estimar fenología.